# 🤖 Oracle Trader v2 — Notebook de Treinamento HMM + PPO

Pipeline completo de Reinforcement Learning para trading:
1. **HMM** — Detecção de regime de mercado (Gaussian HMM)
2. **PPO** — Decisão de trade com GPU (Stable-Baselines3)
3. **Backtest** — Avaliação out-of-sample
4. **Export** — ZIP formato v2.0 (metadata no `zip.comment`)

---

**Novidades v2 Notebook 1.0:**
- Download direto da **API cTrader** (substitui CSV do Supabase)
- **Zero inputs interativos** — suporta `Runtime → Run All` (Ctrl+F9)
- Parâmetros do símbolo (point, spread, commission) obtidos **da API**
- Período do histórico **configurável** (years/months/days)
- Formato de saída **v2.0** — metadata completo no `zip.comment`
- Parâmetros avançados **documentados inline**

**Instruções:**
1. Configure a **SEÇÃO 0** (símbolo, timeframe, período)
2. (Opcional) Ajuste a **SEÇÃO 1** se quiser experimentar
3. Execute: `Runtime → Run All` (Ctrl+F9)
4. Modelo `.zip` será gerado e uploaded para o Supabase


---
## ⚙️ SEÇÃO 0: Configuração Principal

Parâmetros que o usuário **DEVE** configurar para cada treino.

In [ ]:
# =============================================================================
# ⚙️ SEÇÃO 0: CONFIGURAÇÃO PRINCIPAL
# =============================================================================
# Configure os parâmetros abaixo para seu modelo.
# Após configurar, execute: Runtime → Run All (Ctrl+F9)
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# 📊 SÍMBOLO E TIMEFRAME
# ─────────────────────────────────────────────────────────────────────────────
SYMBOL = "EURUSD"              # Nome do símbolo no cTrader (ex: EURUSD, GBPUSD, US500)
TIMEFRAME = "M15"              # Timeframe: M1, M5, M15, M30, H1, H4, D1

# ─────────────────────────────────────────────────────────────────────────────
# 📅 PERÍODO DO HISTÓRICO
# ─────────────────────────────────────────────────────────────────────────────
HISTORY_AMOUNT = 2             # Quantidade de períodos para trás
HISTORY_UNIT = "years"         # Unidade: "years", "months", "days"
HISTORY_END_DATE = None        # Data final (YYYY-MM-DD), None = data atual

# Exemplo: AMOUNT=2, UNIT="years", END_DATE=None
#          → Baixa dados dos últimos 2 anos até hoje

# ─────────────────────────────────────────────────────────────────────────────
# 🔮 PARÂMETROS HMM (Detecção de Regime de Mercado)
# ─────────────────────────────────────────────────────────────────────────────
HMM_STATES = 5                 # Número de estados/regimes (default: 5)
HMM_MOMENTUM_PERIOD = 12       # Período para cálculo de momentum (default: 12)
HMM_CONSISTENCY_PERIOD = 12    # Período para cálculo de consistência (default: 12)
HMM_RANGE_PERIOD = 20          # Período para posição no range (default: 20)

# ─────────────────────────────────────────────────────────────────────────────
# 💰 CUSTOS DE EXECUÇÃO
# ─────────────────────────────────────────────────────────────────────────────
SLIPPAGE_POINTS = 2            # Slippage simulado em pontos (default: 2)

# =============================================================================
# ▶️ Após configurar, execute: Runtime → Run All (Ctrl+F9)
# =============================================================================


---
## 🔬 SEÇÃO 1: Parâmetros Avançados

⚠️ **NÃO RECOMENDADO ALTERAR** — Valores calibrados em extensivos backtests.

In [ ]:
# =============================================================================
# 🔬 SEÇÃO 1: PARÂMETROS AVANÇADOS (NÃO RECOMENDADO ALTERAR)
# =============================================================================

# ─── 📈 PARÂMETROS RL (Features do Modelo PPO) ──────────────────────────────

RL_ROC_PERIOD = 10
# │ Rate of Change — momentum de curto prazo
# │ Menor (5-8): mais sensível a ruído | Maior (12-20): captura tendências
# │ Default: 10

RL_ATR_PERIOD = 14
# │ Average True Range — medida de volatilidade
# │ Menor (7-10): reage rápido | Maior (20-30): suaviza picos
# │ Default: 14 (padrão da indústria)

RL_EMA_PERIOD = 200
# │ Média Móvel Exponencial — tendência de longo prazo
# │ Menor (50-100): médio prazo | Maior (200-300): longo prazo
# │ Default: 200 (referência institucional)

RL_RANGE_PERIOD = 20
# │ Posição no range (high/low) — suporte/resistência de curto prazo
# │ Menor (10-15): range mais apertado | Maior (30-50): menos falsos breakouts
# │ Default: 20

RL_VOLUME_MA_PERIOD = 20
# │ Média móvel de volume — detecção de volume anormal
# │ Em forex, volume = tick volume (proxy, não volume real)
# │ Default: 20

# ─── 🧠 PARÂMETROS DE TREINO PPO ────────────────────────────────────────────

RL_TOTAL_TIMESTEPS = 2_000_000
# │ Passos de treino. Menor (500k): rápido | Maior (5M): risco de overfitting
# │ Default: 2M (~30-60min em GPU T4)

RL_LEARNING_RATE = 3e-4
# │ Taxa de aprendizado. Menor (1e-4): estável | Maior (1e-3): pode oscilar
# │ Default: 3e-4 (recomendado pelo paper PPO)

RL_BATCH_SIZE = 512
# │ Amostras por atualização de gradiente
# │ Default: 512

RL_N_STEPS = 4096
# │ Passos coletados antes de cada atualização (deve ser divisível por BATCH_SIZE)
# │ Default: 4096

RL_GAMMA = 0.99
# │ Fator de desconto. Menor (0.9): foco curto prazo | Maior (0.999): longo prazo
# │ Default: 0.99

# ─── 💵 PARÂMETROS DE TRADING ────────────────────────────────────────────────

INITIAL_BALANCE = 10000        # Balance inicial da conta simulada
COMMISSION_PER_LOT = 7.0       # Comissão por lote round-trip (ECN típico)

# ─── 📊 SPLIT DE DADOS ──────────────────────────────────────────────────────

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# ─── 🔧 OVERRIDE DE CUSTOS (None = usa valor do cTrader) ────────────────────

SPREAD_OVERRIDE = None         # Spread fixo em pontos (None = API)
COMMISSION_OVERRIDE = None     # Comissão fixa por lote (None = default 7.0)

# =============================================================================


---
## 📦 SEÇÃO 2: Setup

Instalação de dependências, detecção de ambiente e imports.

In [ ]:
# =============================================================================
# 📦 SEÇÃO 2.1: Instalação de Dependências
# =============================================================================
!pip install -q hmmlearn gymnasium==0.29.0 stable-baselines3==2.1.0 supabase ctrader-open-api>=0.9.0 twisted==21.7.0 protobuf==3.20.1 service_identity 2>/dev/null


In [ ]:
# =============================================================================
# 📦 SEÇÃO 2.2: Imports e Detecção de Ambiente
# =============================================================================
import os, sys, warnings, contextlib

warnings.filterwarnings('ignore')

# Suprime warnings de deprecação de bibliotecas externas (Python 3.12+)
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='twisted')
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
warnings.filterwarnings('ignore', category=DeprecationWarning, module='pyiceberg')
warnings.filterwarnings('ignore', category=DeprecationWarning, module='supabase')
warnings.filterwarnings('ignore', message='.*utcnow.*')
warnings.filterwarnings('ignore', message='.*utcfromtimestamp.*')
warnings.filterwarnings('ignore', message='.*currentThread.*')
warnings.filterwarnings('ignore', message='.*model_validator.*')
warnings.filterwarnings('ignore', message='.*timeout.*parameter.*deprecated.*')
warnings.filterwarnings('ignore', message='.*verify.*parameter.*deprecated.*')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['GRPC_VERBOSITY'] = 'ERROR'

@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, 'w') as devnull:
        old_stderr, old_stdout = sys.stderr, sys.stdout
        sys.stderr = sys.stdout = devnull
        try: yield
        finally: sys.stderr, sys.stdout = old_stderr, old_stdout

with suppress_output():
    import logging
    logging.getLogger().setLevel(logging.ERROR)
    for name in ['tensorflow', 'absl', 'h5py', 'urllib3', 'google', 'jax']:
        logging.getLogger(name).setLevel(logging.ERROR)
    import numpy as np
    import pandas as pd
    import pickle, json, torch, hashlib, zipfile, time
    import gymnasium as gym
    from gymnasium import spaces
    from hmmlearn import hmm
    from stable_baselines3 import PPO
    from stable_baselines3.common.vec_env import DummyVecEnv
    from stable_baselines3.common.callbacks import BaseCallback

from pathlib import Path
from datetime import datetime, timedelta, timezone
from typing import Tuple, Optional, Dict, List
import matplotlib.pyplot as plt

# ─── Detecção de Ambiente ────────────────────────────────────────────────────
def running_on_kaggle(): return Path("/kaggle").exists()
def running_on_colab(): return "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if running_on_kaggle():
    WORK_DIR = "/kaggle/working"
    ENV_NAME = "Kaggle"
elif running_on_colab():
    WORK_DIR = "/content"
    ENV_NAME = "Colab"
else:
    WORK_DIR = "."
    ENV_NAME = "Local"

os.chdir(WORK_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Ambiente: {ENV_NAME} | Device: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   Working Dir: {WORK_DIR}")


---
## 🔗 SEÇÃO 3: Conexão cTrader + Download de Dados

Conecta na API cTrader, obtém info do símbolo e baixa histórico OHLCV.

In [ ]:
# =============================================================================
# 🔗 SEÇÃO 3.1: Obter Secrets
# =============================================================================
def get_secret(name):
    """Obtém secret do Kaggle ou Colab."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except: pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except: pass
    # Fallback: variável de ambiente
    val = os.environ.get(name)
    if val: return val
    raise ValueError(f"Secret '{name}' não encontrado. Configure nos Secrets do {ENV_NAME}.")

CTRADER_CLIENT_ID = get_secret("CTRADER_CLIENT_ID")
CTRADER_CLIENT_SECRET = get_secret("CTRADER_CLIENT_SECRET")
CTRADER_ACCESS_TOKEN = get_secret("CTRADER_ACCESS_TOKEN")
CTRADER_ACCOUNT_ID = int(get_secret("CTRADER_ACCOUNT_ID"))

# Supabase (para upload do modelo)
SUPABASE_URL = get_secret("SUPABASE_URL")
SUPABASE_KEY = get_secret("SUPABASE_KEY")

print("✓ Secrets carregados")


In [ ]:
# =============================================================================
# 🔗 SEÇÃO 3.2: Conectar cTrader e Baixar Dados
# =============================================================================
from ctrader_open_api import Client, TcpProtocol, EndPoints, Protobuf, Auth
from ctrader_open_api.messages.OpenApiMessages_pb2 import *
from ctrader_open_api.messages.OpenApiModelMessages_pb2 import *
from twisted.internet import reactor
import threading

# ─── Mapeamento de Timeframes ────────────────────────────────────────────────
TIMEFRAME_TO_PERIOD = {
    "M1": 1, "M2": 2, "M3": 3, "M4": 4, "M5": 5, "M10": 6, "M15": 7,
    "M30": 8, "H1": 9, "H4": 10, "H12": 11, "D1": 12, "W1": 13, "MN1": 14,
}

TIMEFRAME_SECONDS = {
    "M1": 60, "M5": 300, "M15": 900, "M30": 1800,
    "H1": 3600, "H4": 14400, "D1": 86400,
}

TIMEFRAME_BARS_PER_YEAR = {
    'M1': 252*20*60, 'M5': 252*20*12, 'M15': 252*20*4,
    'M30': 252*20*2, 'H1': 252*20, 'H4': 252*5, 'D1': 252,
}

# ─── Calcular período ────────────────────────────────────────────────────────
if HISTORY_END_DATE:
    end_date = datetime.strptime(HISTORY_END_DATE, "%Y-%m-%d")
else:
    end_date = datetime.now(timezone.utc)

if HISTORY_UNIT == "years":
    start_date = end_date.replace(year=end_date.year - HISTORY_AMOUNT)
elif HISTORY_UNIT == "months":
    month = end_date.month - HISTORY_AMOUNT
    year = end_date.year
    while month <= 0:
        month += 12
        year -= 1
    start_date = end_date.replace(year=year, month=month)
elif HISTORY_UNIT == "days":
    start_date = end_date - timedelta(days=HISTORY_AMOUNT)
else:
    raise ValueError(f"HISTORY_UNIT inválido: {HISTORY_UNIT}")

from_ms = int(start_date.timestamp() * 1000)
to_ms = int(end_date.timestamp() * 1000)

print(f"📅 Período: {start_date.strftime('%Y-%m-%d')} → {end_date.strftime('%Y-%m-%d')}")
print(f"   Símbolo: {SYMBOL} | Timeframe: {TIMEFRAME}")

# ─── Classe de download via Twisted ──────────────────────────────────────────
class CTraderDataDownloader:
    """Baixa dados históricos via cTrader Open API (Twisted)."""

    def __init__(self, client_id, client_secret, access_token, account_id):
        self.client_id = client_id
        self.client_secret = client_secret
        self.access_token = access_token
        self.account_id = account_id
        self.client = None
        self.bars = []
        self.symbol_info = {}
        self.symbol_id = None
        self._connected = threading.Event()
        self._done = threading.Event()
        self._error = None

    def start(self):
        self.client = Client(EndPoints.PROTOBUF_DEMO_HOST, EndPoints.PROTOBUF_PORT, TcpProtocol)
        self.client.setConnectedCallback(self._on_connected)
        self.client.setDisconnectedCallback(self._on_disconnected)
        self.client.setMessageReceivedCallback(self._on_message)

        # Roda reactor em thread separada
        if not reactor.running:
            self._reactor_thread = threading.Thread(target=reactor.run, kwargs={'installSignalHandlers': False}, daemon=True)
            self._reactor_thread.start()

        reactor.callFromThread(self.client.startService)
        self._connected.wait(timeout=15)

        if not self._connected.is_set():
            raise ConnectionError("Timeout ao conectar no cTrader")

    def _on_connected(self, client):
        print("   TCP/SSL conectado")
        # App Auth
        msg = ProtoOAApplicationAuthReq()
        msg.clientId = self.client_id
        msg.clientSecret = self.client_secret
        d = self.client.send(msg)
        d.addCallback(self._on_app_auth)
        d.addErrback(self._on_error)

    def _on_app_auth(self, response):
        print("   App Auth OK")
        msg = ProtoOAAccountAuthReq()
        msg.accessToken = self.access_token
        msg.ctidTraderAccountId = self.account_id
        d = self.client.send(msg)
        d.addCallback(self._on_account_auth)
        d.addErrback(self._on_error)

    def _on_account_auth(self, response):
        print(f"   Account Auth OK (id={self.account_id})")
        # Carrega símbolos
        msg = ProtoOASymbolsListReq()
        msg.ctidTraderAccountId = self.account_id
        d = self.client.send(msg, responseTimeoutInSeconds=15)
        d.addCallback(self._on_symbols)
        d.addErrback(self._on_error)

    def _on_symbols(self, response):
        res = Protobuf.extract(response)
        symbol_map = {}
        for sym in res.symbol:
            name = sym.symbolName if hasattr(sym, 'symbolName') else str(sym.symbolId)
            symbol_map[name] = sym.symbolId

        if SYMBOL not in symbol_map:
            self._error = f"Símbolo '{SYMBOL}' não encontrado. Disponíveis: {sorted(list(symbol_map.keys())[:20])}"
            self._done.set()
            return

        self.symbol_id = symbol_map[SYMBOL]
        print(f"   Símbolo {SYMBOL} → id={self.symbol_id}")

        # Solicita detalhes do símbolo
        msg = ProtoOASymbolByIdReq()
        msg.ctidTraderAccountId = self.account_id
        msg.symbolId.append(self.symbol_id)
        d = self.client.send(msg, responseTimeoutInSeconds=10)
        d.addCallback(self._on_symbol_details)
        d.addErrback(self._on_error)

    def _on_symbol_details(self, response):
        res = Protobuf.extract(response)
        if res.symbol:
            s = res.symbol[0]
            digits = s.digits if hasattr(s, 'digits') else 5
            pip_pos = s.pipPosition if hasattr(s, 'pipPosition') else (digits - 1)
            self.symbol_info = {
                'point': 10 ** (-digits),
                'digits': digits,
                'pip_value': 10.0,  # Forex standard, será refinado se necessário
                'spread_points': getattr(s, 'spreadMin', 7) if hasattr(s, 'spreadMin') else 7,
                'min_lot': s.minVolume / 10000000.0 if hasattr(s, 'minVolume') else 0.01,
                'max_lot': s.maxVolume / 10000000.0 if hasattr(s, 'maxVolume') else 100.0,
                'lot_step': s.stepVolume / 10000000.0 if hasattr(s, 'stepVolume') else 0.01,
            }
            print(f"   Symbol Info: digits={digits}, spread={self.symbol_info['spread_points']}, min_lot={self.symbol_info['min_lot']}")
        self._connected.set()
        self._download_bars()

    def _download_bars(self):
        """Baixa barras em chunks (API limita ~2000 por request)."""
        print(f"\n📥 Baixando barras {SYMBOL} {TIMEFRAME}...")
        self._all_bars_raw = []
        self._current_from = from_ms
        self._request_next_chunk()

    def _request_next_chunk(self):
        if self._current_from >= to_ms:
            self._finalize_bars()
            return

        chunk_to = min(self._current_from + 604800000, to_ms)  # Max 7 dias por request

        msg = ProtoOAGetTrendbarsReq()
        msg.ctidTraderAccountId = self.account_id
        msg.symbolId = self.symbol_id
        msg.period = TIMEFRAME_TO_PERIOD[TIMEFRAME]
        msg.fromTimestamp = self._current_from
        msg.toTimestamp = chunk_to
        d = self.client.send(msg, responseTimeoutInSeconds=30)
        d.addCallback(self._on_bars_chunk, chunk_to)
        d.addErrback(self._on_error)

    def _on_bars_chunk(self, response, chunk_to):
        res = Protobuf.extract(response)
        count = 0
        if hasattr(res, 'trendbar'):
            for tb in res.trendbar:
                low = tb.low / 100000.0 if tb.low else 0
                bar_time = int(tb.utcTimestampInMinutes * 60) if hasattr(tb, 'utcTimestampInMinutes') else 0
                self._all_bars_raw.append({
                    'time': bar_time,
                    'datetime': datetime.fromtimestamp(bar_time, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S'),
                    'open': low + (tb.deltaOpen / 100000.0 if hasattr(tb, 'deltaOpen') else 0),
                    'high': low + (tb.deltaHigh / 100000.0 if hasattr(tb, 'deltaHigh') else 0),
                    'low': low,
                    'close': low + (tb.deltaClose / 100000.0 if hasattr(tb, 'deltaClose') else 0),
                    'volume': float(tb.volume) if hasattr(tb, 'volume') else 0,
                })
                count += 1

        self._current_from = chunk_to
        total = len(self._all_bars_raw)
        pct = min(100, int((self._current_from - from_ms) / max(1, to_ms - from_ms) * 100))
        print(f"   [{pct:3d}%] {total:,} barras baixadas...", end='\r')

        # Próximo chunk (com rate limit)
        from twisted.internet import reactor as r
        r.callLater(0.3, self._request_next_chunk)

    def _finalize_bars(self):
        self._all_bars_raw.sort(key=lambda b: b['time'])
        # Remove duplicatas
        seen = set()
        unique = []
        for b in self._all_bars_raw:
            if b['time'] not in seen:
                seen.add(b['time'])
                unique.append(b)
        self.bars = unique
        print(f"\n✓ {len(self.bars):,} barras únicas baixadas")
        self._done.set()

    def _on_disconnected(self, client, reason):
        if not self._done.is_set():
            self._error = f"Desconectado: {reason}"
            self._done.set()

    def _on_error(self, failure):
        self._error = str(failure)
        self._done.set()

    def _on_message(self, client, message):
        pass  # Tratado via callbacks específicos

    def wait(self, timeout=300):
        self._done.wait(timeout=timeout)
        if self._error:
            raise RuntimeError(f"Erro cTrader: {self._error}")

    def stop(self):
        if self.client:
            reactor.callFromThread(self.client.stopService)

# ─── Executar download ───────────────────────────────────────────────────────
downloader = CTraderDataDownloader(
    CTRADER_CLIENT_ID, CTRADER_CLIENT_SECRET,
    CTRADER_ACCESS_TOKEN, CTRADER_ACCOUNT_ID
)
downloader.start()
downloader.wait(timeout=300)

# Cria DataFrame
df = pd.DataFrame(downloader.bars)
SYMBOL_CONFIG = downloader.symbol_info

downloader.stop()

print(f"\n📊 DataFrame: {len(df):,} barras | Colunas: {list(df.columns)}")
print(f"   Período: {df['datetime'].iloc[0]} → {df['datetime'].iloc[-1]}")


---
## ⚙️ SEÇÃO 4: Configuração Automática

Deriva parâmetros do símbolo a partir da info obtida da API.

In [ ]:
# =============================================================================
# ⚙️ SEÇÃO 4: Configuração automática baseada no símbolo
# =============================================================================

# ─── Parâmetros do símbolo (da API) ─────────────────────────────────────────
POINT_VALUE = SYMBOL_CONFIG['point']
PIP_VALUE_PER_LOT = SYMBOL_CONFIG.get('pip_value', 10.0)
SPREAD_POINTS = SPREAD_OVERRIDE if SPREAD_OVERRIDE is not None else SYMBOL_CONFIG.get('spread_points', 7)
DIGITS = SYMBOL_CONFIG['digits']
MIN_LOT = SYMBOL_CONFIG.get('min_lot', 0.01)
MAX_LOT = SYMBOL_CONFIG.get('max_lot', 100.0)
LOT_STEP = SYMBOL_CONFIG.get('lot_step', MIN_LOT)
BARS_PER_YEAR = TIMEFRAME_BARS_PER_YEAR.get(TIMEFRAME, 20160)

if COMMISSION_OVERRIDE is not None:
    COMMISSION_PER_LOT = COMMISSION_OVERRIDE

# ─── LOT_SIZES dinâmico ─────────────────────────────────────────────────────
def round_lot(size, step):
    return round(size / step) * step

LOT_SIZES = [
    0,                                              # FLAT (intensity 0)
    round_lot(MIN_LOT, LOT_STEP),                   # WEAK (intensity 1)
    round_lot(MIN_LOT * 3, LOT_STEP),               # MODERATE (intensity 2)
    round_lot(min(MIN_LOT * 5, MAX_LOT), LOT_STEP), # STRONG (intensity 3)
]

# ─── Ações v2.0 (compatível com oracle_trader_v2/core/actions.py) ───────────
ACTIONS = {
    0: {'name': 'WAIT',           'direction': 0,  'intensity': 0, 'size': 0},
    1: {'name': 'LONG_WEAK',      'direction': 1,  'intensity': 1, 'size': LOT_SIZES[1]},
    2: {'name': 'LONG_MODERATE',  'direction': 1,  'intensity': 2, 'size': LOT_SIZES[2]},
    3: {'name': 'LONG_STRONG',    'direction': 1,  'intensity': 3, 'size': LOT_SIZES[3]},
    4: {'name': 'SHORT_WEAK',     'direction': -1, 'intensity': 1, 'size': LOT_SIZES[1]},
    5: {'name': 'SHORT_MODERATE', 'direction': -1, 'intensity': 2, 'size': LOT_SIZES[2]},
    6: {'name': 'SHORT_STRONG',   'direction': -1, 'intensity': 3, 'size': LOT_SIZES[3]},
}

# ─── Nomes derivados ────────────────────────────────────────────────────────
SYMBOL_CLEAN = SYMBOL.replace('.', '').replace('#', '').replace('Cash', '').upper()
base_name = f"{SYMBOL_CLEAN}_{TIMEFRAME}"

print(f"✓ Configuração automática")
print(f"   Symbol: {SYMBOL} → {SYMBOL_CLEAN}")
print(f"   Spread: {SPREAD_POINTS} pts | Slippage: {SLIPPAGE_POINTS} pts | Commission: {COMMISSION_PER_LOT}/lot")
print(f"   LOT_SIZES: {LOT_SIZES} (min={MIN_LOT}, max={MAX_LOT}, step={LOT_STEP})")
print(f"   Bars/Year: {BARS_PER_YEAR:,} | Actions: {len(ACTIONS)}")


---
## 📊 SEÇÃO 5: Visualização dos Dados

In [ ]:
# =============================================================================
# 📊 SEÇÃO 5: Visualização dos Dados
# =============================================================================
print(f"📊 {SYMBOL} {TIMEFRAME}: {len(df):,} barras")
print(f"   Período: {df['datetime'].iloc[0]} → {df['datetime'].iloc[-1]}")
print(f"   Close: min={df['close'].min():.5f}, max={df['close'].max():.5f}, last={df['close'].iloc[-1]:.5f}")

plt.figure(figsize=(14, 4))
plt.plot(df['close'].values, linewidth=0.5)
plt.title(f'{SYMBOL} {TIMEFRAME} — Close Price ({len(df):,} barras)')
plt.ylabel('Price')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 🔮 SEÇÃO 6: Treino HMM — Detecção de Regime 🔒

⚠️ **INTOCÁVEL** — Lógica validada e idêntica ao pipeline de produção.

In [ ]:
# =============================================================================
# 🔮 SEÇÃO 6.1: Features HMM  🔒 INTOCÁVEL
# =============================================================================

def calc_momentum(close, period=10):
    return (close.pct_change().rolling(period).sum() * 100.0).clip(-5.0, 5.0)

def calc_consistency(close, period=10):
    returns = close.pct_change()
    up = (returns > 0).rolling(period).sum()
    down = (returns < 0).rolling(period).sum()
    return (np.maximum(up, down) / period * 2.0 - 1.0) * np.sign(up - down)

def calc_range_position(close, high, low, period=20):
    highest, lowest = high.rolling(period).max(), low.rolling(period).min()
    rng = (highest - lowest).replace(0, np.nan)
    return (close - lowest) / rng * 2.0 - 1.0

df['hmm_momentum'] = calc_momentum(df['close'], HMM_MOMENTUM_PERIOD)
df['hmm_consistency'] = calc_consistency(df['close'], HMM_CONSISTENCY_PERIOD)
df['hmm_range_pos'] = calc_range_position(df['close'], df['high'], df['low'], HMM_RANGE_PERIOD)
df_clean = df.dropna().reset_index(drop=True)
print(f"✓ Features HMM calculadas: {len(df_clean):,} amostras")


In [ ]:
# =============================================================================
# 🔮 SEÇÃO 6.2: Treino HMM  🔒 INTOCÁVEL
# =============================================================================

X_hmm = df_clean[['hmm_momentum', 'hmm_consistency', 'hmm_range_pos']].values
n = len(df_clean)
train_end = int(n * TRAIN_RATIO)
X_train = X_hmm[:train_end]

print(f"Treinando HMM ({HMM_STATES} estados) — Train: {len(X_train):,} amostras")
hmm_model = hmm.GaussianHMM(n_components=HMM_STATES, covariance_type="full", n_iter=100, random_state=42)
hmm_model.fit(X_train)
print(f"✓ Convergiu: {hmm_model.monitor_.converged} | Log-likelihood: {hmm_model.score(X_train):.2f}")


In [ ]:
# =============================================================================
# 🔮 SEÇÃO 6.3: Análise dos Estados HMM  🔒 INTOCÁVEL
# =============================================================================

states = hmm_model.predict(X_hmm)
df_clean['hmm_state'] = states

print("\n" + "="*60 + "\nANÁLISE DOS ESTADOS HMM\n" + "="*60)
state_info = []
for i in range(HMM_STATES):
    pct = 100 * (states == i).sum() / len(states)
    m, c, r = hmm_model.means_[i]
    label = "BULL" if m > 0.1 and r > 0.2 else ("BEAR" if m < -0.1 and r < -0.2 else "RANGE")
    state_info.append({'state': i, 'pct': pct, 'momentum': m, 'consistency': c, 'range_pos': r, 'label': label})
    print(f"State {i}: {pct:.1f}% — {label} (mom={m:+.3f}, cons={c:+.3f}, rng={r:+.3f})")

bull_states = [s['state'] for s in state_info if s['label'] == 'BULL']
bear_states = [s['state'] for s in state_info if s['label'] == 'BEAR']
range_states = [s['state'] for s in state_info if s['label'] == 'RANGE']
print(f"\nBULL: {bull_states} | BEAR: {bear_states} | RANGE: {range_states}")


---
## 🧠 SEÇÃO 7: Treino PPO — Modelo de Decisão 🔒

⚠️ **INTOCÁVEL** — Lógica validada e idêntica ao pipeline de produção.

In [ ]:
# =============================================================================
# 🧠 SEÇÃO 7.1: Features RL  🔒 INTOCÁVEL
# =============================================================================

def calc_roc(close, period=10):
    return np.tanh((close - close.shift(period)) / close.shift(period) * 20)

def calc_atr_norm(h, l, c, period=14):
    tr = pd.concat([h-l, abs(h-c.shift(1)), abs(l-c.shift(1))], axis=1).max(axis=1)
    return np.tanh((tr.rolling(period).mean() / c) * 50)

def calc_trend(close, ema_period=200):
    ema = close.ewm(span=ema_period, adjust=False).mean()
    return np.tanh(((close - ema) / ema) * 20)

def calc_volume_rel(vol, period=20):
    return np.tanh((vol / vol.rolling(period).mean() - 1) * 2)

def calc_session(dt):
    if dt.dtype == 'object': dt = pd.to_datetime(dt)
    return np.sin(2 * np.pi * dt.dt.hour / 24)

df_clean['rl_momentum'] = calc_roc(df_clean['close'], RL_ROC_PERIOD)
df_clean['rl_volatility'] = calc_atr_norm(df_clean['high'], df_clean['low'], df_clean['close'], RL_ATR_PERIOD)
df_clean['rl_trend'] = calc_trend(df_clean['close'], RL_EMA_PERIOD)
df_clean['rl_range_pos'] = calc_range_position(df_clean['close'], df_clean['high'], df_clean['low'], RL_RANGE_PERIOD)
df_clean['rl_volume'] = calc_volume_rel(df_clean['volume'], RL_VOLUME_MA_PERIOD) if 'volume' in df_clean.columns and df_clean['volume'].sum() > 0 else 0
df_clean['rl_session'] = calc_session(df_clean['datetime']) if 'datetime' in df_clean.columns else 0
for i in range(HMM_STATES):
    df_clean[f'hmm_state_{i}'] = (df_clean['hmm_state'] == i).astype(float)

df_rl = df_clean.dropna().reset_index(drop=True)
feature_cols = ['rl_momentum', 'rl_volatility', 'rl_trend', 'rl_range_pos', 'rl_volume', 'rl_session'] + [f'hmm_state_{i}' for i in range(HMM_STATES)]
print(f"✓ Features RL: {len(df_rl):,} amostras, {len(feature_cols)} features")


In [ ]:
# =============================================================================
# 🧠 SEÇÃO 7.2: TradingEnv  🔒 INTOCÁVEL
# =============================================================================
# ⚠️ REGRA DE OURO: Esta lógica DEVE ser idêntica ao VirtualPositionManager
# do oracle_trader_v2. Não otimize, não refatore, não "melhore".
# =============================================================================

class TradingEnv(gym.Env):
    """Ambiente de trading com spread, slippage e comissão."""

    def __init__(self, df, feature_columns, initial_balance=10000, spread_points=10,
                 slippage_points=2, commission_per_lot=7.0, point_value=0.00001,
                 pip_value_per_lot=10.0, digits=5):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.feature_columns = feature_columns
        self.initial_balance = initial_balance
        self.spread_points = spread_points
        self.slippage_points = slippage_points
        self.commission_per_lot = commission_per_lot
        self.point_value = point_value
        self.pip_value_per_lot = pip_value_per_lot
        self.digits = digits
        self.points_per_pip = 10 if digits in [5, 3] else 1

        self.action_space = spaces.Discrete(len(ACTIONS))
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(len(feature_columns) + 3,), dtype=np.float32)

        self.features = self.df[feature_columns].values.astype(np.float32)
        self.closes = self.df['close'].values
        self.hmm_states = self.df['hmm_state'].values if 'hmm_state' in self.df.columns else np.zeros(len(self.df))
        self.max_steps = len(self.df) - 1
        self.reset()

    def _get_slippage(self):
        return np.random.uniform(0, self.slippage_points) * self.point_value

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.balance = self.initial_balance
        self.position_direction = self.position_hmm_state = 0
        self.position_intensity = 0
        self.position_size = self.position_price = self.total_pnl = 0.0
        self.trades = []
        return self._get_obs(), {}

    def step(self, action):
        price = self.closes[self.current_step]
        hmm_st = int(self.hmm_states[self.current_step])
        act = ACTIONS[int(action)]
        reward = self._execute_action(act['direction'], act.get('intensity', 0), act['size'], price, hmm_st)
        self.current_step += 1
        done = self.current_step >= self.max_steps
        if done and self.position_direction != 0:
            reward += self._close_position(self.closes[self.current_step - 1])
        return self._get_obs(), reward, done, False, {
            'step': self.current_step, 'balance': self.balance,
            'position': self.position_direction, 'total_pnl': self.total_pnl,
            'n_trades': len(self.trades)
        }

    def _execute_action(self, target_dir, target_intensity, target_size, price, hmm_st):
        # Mesma posição e intensidade → NOOP
        if target_dir == self.position_direction and target_intensity == self.position_intensity:
            return 0.0
        reward = self._close_position(price) if self.position_direction != 0 else 0.0
        if target_dir != 0 and target_size > 0:
            self._open_position(target_dir, target_intensity, target_size, price, hmm_st)
        return reward

    def _open_position(self, direction, intensity, size, price, hmm_st):
        spread = self.spread_points * self.point_value
        slip = self._get_slippage()
        self.position_price = price + spread + slip if direction == 1 else price - spread - slip
        self.position_direction = direction
        self.position_intensity = intensity
        self.position_size = size
        self.position_hmm_state = hmm_st
        self.balance -= (self.commission_per_lot * size) / 2

    def _close_position(self, price):
        if self.position_direction == 0: return 0.0
        slip = self._get_slippage()
        exit_price = price - slip if self.position_direction == 1 else price + slip
        price_diff = (exit_price - self.position_price) * self.position_direction
        pips = price_diff / self.point_value / self.points_per_pip
        pnl = pips * self.pip_value_per_lot * self.position_size - (self.commission_per_lot * self.position_size) / 2
        self.balance += pnl
        self.total_pnl += pnl
        self.trades.append({
            'direction': self.position_direction, 'intensity': self.position_intensity,
            'size': self.position_size, 'pnl': pnl, 'pips': pips,
            'hmm_state': self.position_hmm_state, 'step': self.current_step
        })
        self.position_direction = self.position_intensity = 0
        self.position_size = self.position_price = self.position_hmm_state = 0
        return pnl

    def _get_obs(self):
        market = self.features[self.current_step]
        floating = 0.0
        if self.position_direction != 0:
            floating = np.tanh((
                (self.closes[self.current_step] - self.position_price)
                * self.position_direction
                / self.point_value / self.points_per_pip
                * self.pip_value_per_lot * self.position_size
            ) / 100)
        return np.nan_to_num(np.concatenate([
            market, [self.position_direction, self.position_size * 10, floating]
        ]).astype(np.float32))

print(f"✓ TradingEnv definido (spread={SPREAD_POINTS}pts, slippage=0-{SLIPPAGE_POINTS}pts)")


In [ ]:
# =============================================================================
# 🧠 SEÇÃO 7.3: Split + Treino PPO  🔒 INTOCÁVEL
# =============================================================================

n = len(df_rl)
train_end, val_end = int(n * TRAIN_RATIO), int(n * (TRAIN_RATIO + VAL_RATIO))
df_train = df_rl.iloc[:train_end].copy()
df_val = df_rl.iloc[train_end:val_end].copy()
df_test = df_rl.iloc[val_end:].copy()
print(f"📊 Split: Train={len(df_train):,} | Val={len(df_val):,} | Test={len(df_test):,}")

env_params = {
    'initial_balance': INITIAL_BALANCE, 'spread_points': SPREAD_POINTS,
    'slippage_points': SLIPPAGE_POINTS, 'commission_per_lot': COMMISSION_PER_LOT,
    'point_value': POINT_VALUE, 'pip_value_per_lot': PIP_VALUE_PER_LOT, 'digits': DIGITS
}
train_env = TradingEnv(df_train, feature_cols, **env_params)
val_env = TradingEnv(df_val, feature_cols, **env_params)
test_env = TradingEnv(df_test, feature_cols, **env_params)

# ─── Callback de Validação ───────────────────────────────────────────────────
class TradingCallback(BaseCallback):
    def __init__(self, eval_env, check_freq=10000, save_path='best_model', verbose=1):
        super().__init__(verbose)
        self.eval_env, self.check_freq, self.save_path = eval_env, check_freq, save_path
        self.best_reward, self.history = -np.inf, []

    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            rewards = []
            for _ in range(3):
                obs, done, total = self.eval_env.reset()[0], False, 0
                while not done:
                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, reward, done, _, _ = self.eval_env.step(action)
                    total += reward
                rewards.append(total)
            mean_r = np.mean(rewards)
            self.history.append({'step': self.n_calls, 'reward': mean_r})
            if mean_r > self.best_reward:
                diff, self.best_reward = mean_r - self.best_reward, mean_r
                if self.verbose: print(f"  Step {self.n_calls:,}: ${mean_r:.2f} (Best! +${diff:.2f})")
                self.model.save(self.save_path)
            elif self.verbose:
                print(f"  Step {self.n_calls:,}: ${mean_r:.2f}")
        return True

# ─── Treino ──────────────────────────────────────────────────────────────────
print(f"\n{'='*60}\n🧠 TREINAMENTO PPO\n{'='*60}")
print(f"Device: {device} | Steps: {RL_TOTAL_TIMESTEPS:,} | Batch: {RL_BATCH_SIZE}")

vec_env = DummyVecEnv([lambda: train_env])
model = PPO(
    "MlpPolicy", vec_env,
    learning_rate=RL_LEARNING_RATE, n_steps=RL_N_STEPS,
    batch_size=RL_BATCH_SIZE, gamma=RL_GAMMA,
    policy_kwargs=dict(net_arch=dict(pi=[256, 256, 256], vf=[256, 256, 256])),
    verbose=0, device=device
)
callback = TradingCallback(val_env, check_freq=20000, save_path=f'{base_name}_ppo_best')

start = datetime.now()
model.learn(total_timesteps=RL_TOTAL_TIMESTEPS, callback=callback, progress_bar=True)
print(f"\n✓ Treino: {datetime.now() - start} | Best Val: ${callback.best_reward:.2f}")


---
## 📈 SEÇÃO 8: Backtest Out-of-Sample

In [ ]:
# =============================================================================
# 📈 SEÇÃO 8.1: Executar Backtest
# =============================================================================

# Carrega melhor modelo
best_path = f"{base_name}_ppo_best.zip"
if os.path.exists(best_path):
    model = PPO.load(best_path, device=device)
    print(f"✓ Best model carregado: {best_path}")

print(f"\n{'='*60}\n📊 BACKTEST OOS\n{'='*60}")
obs, done, history = test_env.reset()[0], False, []
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _, info = test_env.step(action)
    history.append({'step': info['step'], 'balance': info['balance'], 'position': info['position'], 'reward': reward})
hist_df, trades = pd.DataFrame(history), test_env.trades
print(f"✓ {len(hist_df)} steps, {len(trades)} trades")


In [ ]:
# =============================================================================
# 📈 SEÇÃO 8.2: Métricas
# =============================================================================

def calculate_metrics(trades, hist_df, initial_balance=10000, bars_per_year=20160):
    if not trades: return {'total_trades': 0, 'total_pnl': 0}
    pnls = [t['pnl'] for t in trades]
    winners, losers = [p for p in pnls if p > 0], [p for p in pnls if p < 0]
    balance = hist_df['balance'].values
    peak = np.maximum.accumulate(balance)
    dd = (peak - balance) / peak
    returns = hist_df['reward'].values
    sharpe = np.mean(returns) / np.std(returns) * np.sqrt(bars_per_year) if np.std(returns) > 0 else 0
    neg_ret = returns[returns < 0]
    sortino = np.mean(returns) / np.std(neg_ret) * np.sqrt(bars_per_year) if len(neg_ret) > 0 and np.std(neg_ret) > 0 else sharpe
    total_return = (balance[-1] - initial_balance) / initial_balance
    longs = [t for t in trades if t['direction'] == 1]
    shorts = [t for t in trades if t['direction'] == -1]
    return {
        'total_trades': len(trades), 'total_pnl': sum(pnls),
        'total_pips': sum(t.get('pips', 0) for t in trades),
        'total_return_pct': total_return * 100,
        'win_rate': len(winners) / len(trades),
        'profit_factor': sum(winners) / abs(sum(losers)) if losers else float('inf'),
        'max_drawdown_pct': dd.max() * 100,
        'sharpe_ratio': sharpe, 'sortino_ratio': sortino,
        'calmar_ratio': total_return / dd.max() if dd.max() > 0 else float('inf'),
        'avg_trade': sum(pnls) / len(trades),
        'avg_winner': np.mean(winners) if winners else 0,
        'avg_loser': np.mean(losers) if losers else 0,
        'final_balance': balance[-1],
        'long_trades': len(longs), 'short_trades': len(shorts),
        'long_pnl': sum(t['pnl'] for t in longs),
        'short_pnl': sum(t['pnl'] for t in shorts),
        'long_win_rate': len([t for t in longs if t['pnl'] > 0]) / len(longs) if longs else 0,
        'short_win_rate': len([t for t in shorts if t['pnl'] > 0]) / len(shorts) if shorts else 0,
    }

metrics = calculate_metrics(trades, hist_df, INITIAL_BALANCE, BARS_PER_YEAR)
print(f"\n{'='*60}\n📊 RESULTADOS OOS — {SYMBOL}\n{'='*60}")
print(f"💰 P/L: ${metrics['total_pnl']:.2f} | Pips: {metrics['total_pips']:.1f} | Return: {metrics['total_return_pct']:.2f}%")
print(f"📈 Trades: {metrics['total_trades']} | WR: {metrics['win_rate']*100:.1f}% | PF: {metrics['profit_factor']:.2f}")
print(f"📊 Long: {metrics['long_trades']} (${metrics['long_pnl']:.2f}) | Short: {metrics['short_trades']} (${metrics['short_pnl']:.2f})")
print(f"⚠️ MaxDD: {metrics['max_drawdown_pct']:.2f}% | Sharpe: {metrics['sharpe_ratio']:.2f} | Sortino: {metrics['sortino_ratio']:.2f}")

# Análise por regime
def analyze_by_regime(trades, bull_states, bear_states, range_states):
    def regime(s): return 'BULL' if s in bull_states else ('BEAR' if s in bear_states else 'RANGE')
    results = {}
    for r in ['BULL', 'BEAR', 'RANGE']:
        rt = [t for t in trades if regime(t.get('hmm_state', -1)) == r]
        pnls = [t['pnl'] for t in rt] if rt else []
        results[r] = {'trades': len(rt), 'pnl': sum(pnls), 'win_rate': len([p for p in pnls if p > 0]) / len(rt) if rt else 0, 'avg': np.mean(pnls) if pnls else 0}
    return results

regime_analysis = analyze_by_regime(trades, bull_states, bear_states, range_states)
print(f"\n{'='*60}\n📊 POR REGIME HMM\n{'='*60}")
print(f"{'Regime':<8} {'Trades':<8} {'P/L':<12} {'WR':<10} {'Avg'}")
for r, d in regime_analysis.items():
    print(f"{r:<8} {d['trades']:<8} ${d['pnl']:<11.2f} {d['win_rate']*100:<9.1f}% ${d['avg']:.2f}")


In [ ]:
# =============================================================================
# 📈 SEÇÃO 8.3: Gráficos
# =============================================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Equity curve
axes[0].plot(hist_df['balance'].values, lw=1)
axes[0].axhline(INITIAL_BALANCE, color='gray', ls='--', alpha=0.5)
axes[0].set_ylabel('Balance ($)')
axes[0].set_title(f'{SYMBOL} — Equity Curve OOS')
axes[0].grid(True, alpha=0.3)

# Drawdown
dd = (np.maximum.accumulate(hist_df['balance'].values) - hist_df['balance'].values) / np.maximum.accumulate(hist_df['balance'].values) * 100
axes[1].fill_between(range(len(dd)), 0, -dd, alpha=0.7, color='red')
axes[1].set_ylabel('DD (%)')
axes[1].set_title('Drawdown')
axes[1].grid(True, alpha=0.3)

# P/L por trade
if trades:
    pnls = [t['pnl'] for t in trades]
    axes[2].bar(range(len(pnls)), pnls, color=['g' if p > 0 else 'r' for p in pnls], alpha=0.7)
    axes[2].axhline(0, color='k', lw=0.5)
    axes[2].set_ylabel('P/L ($)')
    axes[2].set_xlabel('Trade #')
    axes[2].set_title('P/L por Trade')
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Resumo final
print(f"\n{'='*60}\n📋 RESUMO FINAL — {SYMBOL}\n{'='*60}")
print(f"Dados: {len(df):,} barras | Teste: {len(df_test):,}")
print(f"Config: spread={SPREAD_POINTS}pts, slip={SLIPPAGE_POINTS}pts")
print(f"HMM: Bull={bull_states}, Bear={bear_states}")
print(f"OOS: P/L=${metrics['total_pnl']:.2f}, WR={metrics['win_rate']*100:.1f}%, DD={metrics['max_drawdown_pct']:.2f}%, Sharpe={metrics['sharpe_ratio']:.2f}")
status = "✅ EXCELENTE" if metrics['total_pnl'] > 0 and metrics['sharpe_ratio'] > 1.0 and metrics['max_drawdown_pct'] < 20 else ("⚠️ BOM" if metrics['total_pnl'] > 0 and metrics['sharpe_ratio'] > 0.5 else ("⚠️ MARGINAL" if metrics['total_pnl'] > 0 else "❌ NEGATIVO"))
print(f"\n{status}")


---
## 📦 SEÇÃO 9: Export & Upload

Salva modelos no formato ZIP v2.0 com metadata no `zip.comment` e faz upload para Supabase.

In [ ]:
# =============================================================================
# 📦 SEÇÃO 9.1: Salvar Modelos Localmente
# =============================================================================

hmm_path = f"{base_name}_hmm.pkl"
ppo_path = f"{base_name}_ppo.zip"

# Salva HMM
with open(hmm_path, 'wb') as f:
    pickle.dump({
        'model': hmm_model,
        'state_mapping': {'bull_states': bull_states, 'bear_states': bear_states, 'range_states': range_states},
        'n_states': HMM_STATES,
        'hmm_params': {'momentum_period': HMM_MOMENTUM_PERIOD, 'consistency_period': HMM_CONSISTENCY_PERIOD, 'range_period': HMM_RANGE_PERIOD}
    }, f)

# Salva PPO
model.save(ppo_path)

# Salva métricas CSV
metrics['symbol'], metrics['timeframe'] = SYMBOL, TIMEFRAME
pd.DataFrame([metrics]).to_csv(f"{base_name}_metrics.csv", index=False)

print(f"✓ Salvos: {hmm_path}, {ppo_path}, {base_name}_metrics.csv")


In [ ]:
# =============================================================================
# 📦 SEÇÃO 9.2: Criar ZIP v2.0 (metadata no zip.comment)
# =============================================================================
# Este formato é lido pelo ModelLoader do oracle_trader_v2
# =============================================================================

zip_filename = f"{base_name}.zip"
model_files = [hmm_path, ppo_path]

# ─── Metadata v2.0 (compatível com SPEC_PREDITOR.md) ────────────────────────
metadata_v2 = {
    "format_version": "2.0",
    "generated_at": datetime.now(timezone.utc).isoformat() + "Z",

    "symbol": {
        "name": SYMBOL,
        "clean": SYMBOL_CLEAN,
        "timeframe": TIMEFRAME
    },

    "training_config": {
        "point": POINT_VALUE,
        "pip_value": PIP_VALUE_PER_LOT,
        "spread_points": SPREAD_POINTS,
        "slippage_points": SLIPPAGE_POINTS,
        "commission_per_lot": COMMISSION_PER_LOT,
        "digits": DIGITS,
        "initial_balance": INITIAL_BALANCE,
        "lot_sizes": LOT_SIZES,
        "total_timesteps": RL_TOTAL_TIMESTEPS
    },

    "hmm_config": {
        "n_states": HMM_STATES,
        "momentum_period": HMM_MOMENTUM_PERIOD,
        "consistency_period": HMM_CONSISTENCY_PERIOD,
        "range_period": HMM_RANGE_PERIOD
    },

    "rl_config": {
        "roc_period": RL_ROC_PERIOD,
        "atr_period": RL_ATR_PERIOD,
        "ema_period": RL_EMA_PERIOD,
        "range_period": RL_RANGE_PERIOD,
        "volume_ma_period": RL_VOLUME_MA_PERIOD
    },

    "actions": {
        str(k): {"name": v["name"], "direction": v["direction"], "intensity": v["intensity"]}
        for k, v in ACTIONS.items()
    },

    "hmm_state_analysis": {
        "bull_states": bull_states,
        "bear_states": bear_states,
        "range_states": range_states
    },

    "backtest_oos": {
        "total_trades": metrics.get('total_trades', 0),
        "total_pnl": round(metrics.get('total_pnl', 0), 2),
        "win_rate": round(metrics.get('win_rate', 0), 3),
        "profit_factor": round(metrics.get('profit_factor', 0), 2),
        "max_drawdown_pct": round(metrics.get('max_drawdown_pct', 0), 2),
        "sharpe_ratio": round(metrics.get('sharpe_ratio', 0), 2),
        "sortino_ratio": round(metrics.get('sortino_ratio', 0), 2),
    },

    "data_info": {
        "total_bars": len(df),
        "train_bars": len(df_train),
        "val_bars": len(df_val),
        "test_bars": len(df_test),
        "date_start": df['datetime'].iloc[0],
        "date_end": df['datetime'].iloc[-1]
    },

    "preditor": {
        "min_bars": 350
    }
}

# ─── Cria ZIP ────────────────────────────────────────────────────────────────
print(f"📦 Criando {zip_filename} (formato v2.0)...")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in model_files:
        zf.write(f)
        print(f"  + {f}")
    # Metadata no zip.comment
    zf.comment = json.dumps(metadata_v2, indent=2).encode('utf-8')

zip_size = Path(zip_filename).stat().st_size
print(f"✓ Zip: {zip_size / 1024 / 1024:.2f} MB")
print(f"✓ Metadata v2.0 gravado no zip.comment ({len(zf.comment)} bytes)")

# Verifica que metadata é legível
with zipfile.ZipFile(zip_filename, 'r') as zf:
    check = json.loads(zf.comment.decode('utf-8'))
    assert check['format_version'] == '2.0'
    print(f"✓ Verificação: format_version={check['format_version']}, symbol={check['symbol']['name']}")


In [ ]:
# =============================================================================
# 📦 SEÇÃO 9.3: Upload para Supabase + Verificação
# =============================================================================

from supabase import create_client
sb_client = create_client(SUPABASE_URL, SUPABASE_KEY)

# Hash local
hash_md5 = hashlib.md5()
with open(zip_filename, 'rb') as f:
    for chunk in iter(lambda: f.read(4096), b""):
        hash_md5.update(chunk)
local_hash = hash_md5.hexdigest()

# Upload
print(f"\n📤 Enviando para oracle_models/{base_name}/...")
with open(zip_filename, 'rb') as f:
    zip_data = f.read()

remote_path = f"{base_name}/{zip_filename}"

try:
    sb_client.storage.from_("oracle_models").upload(
        path=remote_path, file=zip_data,
        file_options={"content-type": "application/zip", "upsert": "true"}
    )
    print("✓ Upload concluído")

    # Verifica hash remoto
    print("🔍 Verificando integridade...", end=" ")
    remote_data = sb_client.storage.from_("oracle_models").download(remote_path)
    remote_hash = hashlib.md5(remote_data).hexdigest()

    if local_hash == remote_hash:
        print(f"✓ OK ({local_hash[:8]}...)")

        # Metadados já estão no zip.comment (formato v2.0), não precisa tabela
        print("✓ Metadados incorporados no zip.comment")

        # Limpeza
        print("\n🧹 Limpando arquivos temporários...")
        for f in [hmm_path, ppo_path, f"{base_name}_metrics.csv", zip_filename, f"{base_name}_ppo_best.zip"]:
            p = Path(f)
            if p.exists():
                p.unlink()
                print(f"  🗑 {f}")

        print(f"\n{'='*50}")
        print(f"🎉 Modelo publicado com sucesso!")
        print(f"   Path: oracle_models/{remote_path}")
        print(f"   Format: v2.0 (metadata in zip.comment)")
        print(f"   Hash: {local_hash[:8]}...")
    else:
        print(f"✗ FALHOU! Local={local_hash} Remoto={remote_hash}")
        print("   Arquivos NÃO foram limpos. Tente novamente.")

except Exception as e:
    print(f"✗ Erro: {e}")
